In [ ]:
# Thay vì import requests thường, ta dùng cái này
from curl_cffi import requests 
import re
import time
import random
import os
import csv

In [ ]:
# --- CẤU HÌNH ---
START_ID = 129038158
END_ID = 129038158
CSV_FILE = "../../data/raw/chotot_raw.csv"


if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        headers = [
            "ID", "IRL (Link gốc)", "Web", "Name", "Furniture", 
            "Money (Price)", "Size", "Address", "Deposit", "Body"
        ]
        writer.writerow(headers)
# Headers đầy đủ hơn để giống người thật
REQUEST_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.nhatot.com/',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'same-origin',
    'Upgrade-Insecure-Requests': '1',
}

VALIDATION_STRINGS = [
    '{"@type":"ListItem","position":2,"name":"Thuê Nhà ở","item":"https://www.nhatot.com/thue-nha-dat"}',
    '{"@type":"ListItem","position":2,"name":"Thuê Phòng trọ","item":"https://www.nhatot.com/thue-phong-tro"}',
    '{"@type":"ListItem","position":2,"name":"Thuê Căn hộ Chung cư","item":"https://www.nhatot.com/thue-can-ho-chung-cu"}'
]

print("🚀 Bắt đầu quá trình đào dữ liệu (Đã Fix 403)...")

In [ ]:
def extract_by_regex(pattern, text):
    try:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            return match.group(1).strip()
    except Exception:
        return None
    return ""

def extract_all_by_regex(pattern, text):
    """Hàm tìm TẤT CẢ các kết quả khớp regex"""
    try:
        # re.findall trả về một LIST các chuỗi khớp với ngoặc đơn (.*?)
        # re.DOTALL: cho phép tìm xuyên qua dấu xuống dòng
        return re.findall(pattern, text, re.DOTALL)
    except Exception:
        return [] # Trả về list rỗng nếu lỗi
def Crawl(x):
    print(f"--- Đang xử lý ID: {x} ---")
    data_chotot = {
        "id": x, "irl": "", "web": "nhatot.com", "name": "", "furniture": "", 
        "money": "", "size": "", "address": "", "deposit": "", "body": ""
    }

    # URL này sẽ tự động redirect về link dài
    url_request = f"https://www.nhatot.com/{x}.htm"
    
    # Để test link cụ thể của bạn (Bỏ comment dòng dưới để chạy test 1 link)
    # url_request = "https://www.nhatot.com/thue-phong-tro-quan-binh-tan-tp-ho-chi-minh/128996281.htm"

    try:
        # --- THAY ĐỔI QUAN TRỌNG Ở ĐÂY ---
        # impersonate="chrome120": Giả dạng trình duyệt Chrome bản 120
        response = requests.get(
            url_request, 
            headers=REQUEST_HEADERS, 
            impersonate="chrome120", 
            timeout=15,
            allow_redirects=True
        )

        if response.status_code != 200:
            print(f"⚠️ Vẫn lỗi HTTP {response.status_code}. Server chặn gắt quá.")
            time.sleep(random.uniform(1.5, 2))
            return
            
        html_content = response.text
        real_url = response.url
        print(f"✅ Status: {response.status_code} | URL: {real_url}")

        # --- ĐOẠN VALIDATION CŨ CỦA BẠN ---
        is_valid_html = False
        for valid_str in VALIDATION_STRINGS:
            if valid_str in html_content:
                is_valid_html = True
                break
        
        if not is_valid_html:
            print(f"❌ HTML không chứa thông tin định danh hợp lệ. (Có thể bị captcha hoặc sai trang)")
            time.sleep(2)
            return

        # --- ĐOẠN REGEX CŨ CỦA BẠN ---
        data_chotot["irl"] = real_url
        data_chotot["name"] = extract_by_regex(r'"position":5,"name":"(.*?)"', html_content)
        data_chotot["furniture"] = extract_by_regex(r'"Tình trạng nội thất","value":"(.*?)"', html_content)
        data_chotot["money"] = extract_by_regex(r'"price":(.*?),', html_content)
        data_chotot["size"] = extract_by_regex(r'"size":\{"id":"size","label":"Diện tích","value":"(.*?)"\}', html_content)
        data_chotot["address"] = extract_by_regex(r'\{"id":"address","label":"Địa chỉ","value":"(.*?)"\}', html_content)
        data_chotot["deposit"] = extract_by_regex(r'"deposit":\{"id":"deposit","label":"Số tiền cọc","value":"(.*?)"\}', html_content)
        data_chotot["body"] = extract_by_regex(r'"body":"(.*?)"', html_content)

        # Ghi vào CSV
        row_values = [
            data_chotot["id"], data_chotot["irl"], data_chotot["web"], data_chotot["name"],
            data_chotot["furniture"], data_chotot["money"], data_chotot["size"],
            data_chotot["address"], data_chotot["deposit"], data_chotot["body"]
        ]
        
        with open(CSV_FILE, mode='a', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerow(row_values)
        print("   -> Đã lưu dữ liệu.")

        sleep_time = random.uniform(1, 1.5)
        time.sleep(sleep_time)

    except Exception as e:
        print(f"⚠️ Lỗi ngoại lệ: {e}")
        time.sleep(random.uniform(1.5, 2))
        return

In [ ]:
for i in range(0,999):
    print(i)
    url_request = f"https://www.nhatot.com/thue-phong-tro-tp-ho-chi-minh?page={i}"
    response = requests.get(
                url_request, 
                headers=REQUEST_HEADERS, 
                impersonate="chrome120", 
                timeout=15,
                allow_redirects=True
            )
    html_content = response.text
    with open('READ.txt',mode='r',encoding='utf-8') as f:
        Read=f.read()
    list_urls = extract_all_by_regex(r'ho-chi-minh/(.*?).htm', html_content)
    time.sleep(1)
    for url in list_urls:
        if url in Read:
            continue
        Read=Read+url+"\n"
        Crawl(url)
    with open('READ.txt',mode='w',encoding='utf-8') as f:
        f.write(Read)